# TCGA-OV Multi-Omics Advanced Evidence Notebook

This Kaggle kernel reproduces and summarizes advanced evidence from the public derived dataset:
- predictive benchmarks
- DAG/network pathway outputs
- perturbation and sensitivity analyses
- input-output ablation experiments
- PCA structure summaries


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
DATA_ROOT = "/kaggle/input/tcga-ov-multiomics-network-derived-results"
assert os.path.exists(DATA_ROOT), f"Dataset path not found: {DATA_ROOT}"
DATA_ROOT


## Load Core Tables

In [ ]:
tables = f"{DATA_ROOT}/results/tables"
nets = f"{DATA_ROOT}/results/networks"
figs = f"{DATA_ROOT}/results/figures"
model_benchmark = pd.read_csv(f"{tables}/model_benchmark.csv")
model_benchmark_prot = pd.read_csv(f"{tables}/model_benchmark_protein_matched.csv")
adv_ml = pd.read_csv(f"{tables}/advanced_ml_benchmark.csv")
ablation = pd.read_csv(f"{tables}/input_output_ablation_auc.csv")
perm = pd.read_csv(f"{tables}/permutation_test_auc.csv")
pca_summary = pd.read_csv(f"{tables}/pca_summary.csv")
sens = pd.read_csv(f"{tables}/sensitivity_hub_slope_summary.csv")
sens_grid = pd.read_csv(f"{tables}/sensitivity_perturb_fraction_grid.csv")
pathway = pd.read_csv(f"{tables}/causal_pathway_strength_summary.csv")
network_centrality = pd.read_csv(f"{nets}/network_centrality.csv")
dag_edges = pd.read_csv(f"{nets}/dag_pathways.csv")
sample_summary = pd.read_csv(f"{tables}/sample_matching_summary.csv")
feature_summary = pd.read_csv(f"{tables}/feature_count_summary.csv")
print("Loaded tables successfully")


## Cohort and Feature Overview

In [ ]:
display(sample_summary)
display(feature_summary)


## Benchmark Models (All-Sample and Protein-Matched)

In [ ]:
display(model_benchmark.sort_values("auc", ascending=False))
display(model_benchmark_prot.sort_values("auc", ascending=False))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
mb = model_benchmark.sort_values("auc", ascending=False)
ax[0].barh(mb["model"], mb["auc"])
ax[0].set_title("All-sample AUC")
ax[0].set_xlim(0, 1)
mp = model_benchmark_prot.sort_values("auc", ascending=False)
ax[1].barh(mp["model"], mp["auc"])
ax[1].set_title("Protein-matched AUC")
ax[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()


## Advanced ML and Permutation Evidence

In [ ]:
display(adv_ml.sort_values("auc", ascending=False))
display(perm)
best = adv_ml.sort_values("auc", ascending=False).iloc[0]
print(f"Top model: {best.model}, AUC={best.auc:.3f}")
print(f"Permutation p-value: {perm.loc[0, "p_value_right_tail"]:.4f}")


## Input-Output Parameter Experiments (Ablation)

In [ ]:
ab = ablation.sort_values("auc", ascending=False)
display(ab)
plt.figure(figsize=(10, 5))
sns.barplot(data=ab.head(10), y="blocks", x="auc", color="steelblue")
plt.xlim(0, 1)
plt.title("Top omics block combinations by AUC")
plt.tight_layout()
plt.show()


## Network, DAG Pathway, and Sensitivity

In [ ]:
display(network_centrality.head(15))
display(pathway.sort_values(["n_edges", "mean_abs_weight"], ascending=False))
display(sens.sort_values("delta_global_slope", ascending=False).head(10))
plt.figure(figsize=(7, 5))
top = sens.sort_values("delta_global_slope", ascending=False).head(6)["hub"].tolist()
for h in top:
    d = sens_grid[sens_grid["hub"] == h].sort_values("perturb_fraction")
    plt.plot(d["perturb_fraction"], d["delta_global_pagerank_l1"], marker="o", label=h)
plt.title("Sensitivity curves for top hubs")
plt.xlabel("Perturbation fraction")
plt.ylabel("Delta global PageRank L1")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


## PCA Structure Evidence

In [ ]:
display(pca_summary.sort_values("pc1_pc2_cum_var", ascending=False))


## Final Evidence Snapshot

This notebook confirms that the released TCGA-OV evidence package contains:
- reproducible cross-layer integration outputs
- robust network and perturbation sensitivity evidence
- advanced ML and permutation-based inferential support
- interpretable input-output ablation experiments
